# CUDA GNN inference — Colab runner

This notebook is self-contained. It builds the repository on Colab's local disk, stores benchmark results in Google Drive, validates all compiled backends, and runs a small synthetic GCN/GraphSAGE experiment.

Before running it, select **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU). The public-dataset and million-node experiments near the end are opt-in.

In [ ]:
import shutil
import subprocess
import sys

for tool in ('git', 'g++', 'nvcc', 'nvidia-smi'):
    if shutil.which(tool) is None:
        raise RuntimeError(f'{tool} is unavailable. Select a Colab GPU runtime and reconnect.')

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvcc', '--version'], check=True)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT = Path('/content/cuda-gnn-inference')
RESULTS = Path('/content/drive/MyDrive/cuda-gnn-inference-results')
RESULTS.mkdir(parents=True, exist_ok=True)
print('Persistent results:', RESULTS)

In [ ]:
REPOSITORY = 'https://github.com/Alby02/cuda-gnn-inference.git'

if (PROJECT / '.git').is_dir():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
elif PROJECT.exists():
    raise RuntimeError(f'{PROJECT} exists but is not a Git checkout; restart the runtime or rename it.')
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(PROJECT)], check=True)

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'meson', 'ninja', 'pandas', '-r', str(PROJECT / 'python_libraries.txt'),
], check=True)

In [ ]:
build = PROJECT / 'builddir'
if (build / 'meson-private').is_dir():
    configure = ['meson', 'setup', '--reconfigure', str(build), '-Dopenmp=enabled', '-Dcuda=enabled']
else:
    configure = ['meson', 'setup', str(build), str(PROJECT), '-Dopenmp=enabled', '-Dcuda=enabled']
subprocess.run(configure, check=True)
subprocess.run(['meson', 'compile', '-C', str(build)], check=True)
EXECUTABLE = build / 'gnn'
help_result = subprocess.run([str(EXECUTABLE), '--help'], check=True, text=True, capture_output=True)
print(help_result.stdout)
for required in ('sequential', 'parallel', 'cuda'):
    if required not in help_result.stdout:
        raise RuntimeError(f'Expected backend {required!r} was not compiled')

## Correctness smoke tests

The first test compares the built-in demo across sequential, OpenMP, and CUDA. The second generates tracked fixtures and checks empty, isolated, undirected, weighted, GraphSAGE, GCN, and no-bias paths against the independent Python/PyG reference.

In [ ]:
import re
import numpy as np

def demo_output(backend):
    result = subprocess.run(
        [str(EXECUTABLE), '--backend', backend, '--warmups', '0', '--repetitions', '1'],
        check=True, text=True, capture_output=True, cwd=PROJECT,
    )
    rows = []
    for line in result.stdout.splitlines():
        match = re.search(r'\[([^\]]+)\]', line)
        if match:
            rows.append([float(value) for value in match.group(1).split(',')])
    if not rows:
        raise RuntimeError(f'{backend} produced no parseable output')
    return np.asarray(rows, dtype=np.float32)

sequential = demo_output('sequential')
for backend in ('parallel', 'cuda'):
    np.testing.assert_allclose(demo_output(backend), sequential, rtol=1e-5, atol=1e-5)
print('All native backends match:', sequential.tolist())

In [ ]:
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/check_edge_cases.py'),
    '--native', str(EXECUTABLE),
    '--work-dir', '/content/gnn-edge-cases',
], check=True, cwd=PROJECT)

## Small synthetic end-to-end benchmark

This exercises synthetic graph generation, model export, all native backends, PyTorch Geometric comparison, repeatability checks, CSV output, and plots. It is deliberately small enough for **Run all**.

In [ ]:
QUICK_RESULTS = RESULTS / 'quick-synthetic'
quick_command = [
    sys.executable, str(PROJECT / 'scripts/run_experiments.py'),
    '--native', str(EXECUTABLE),
    '--dataset', 'none',
    '--nodes', '1000',
    '--widths', '32',
    '--depths', '2',
    '--skews', '0',
    '--backend', 'sequential', 'parallel', 'cuda',
    '--threads', '1', '2',
    '--block-size', '128', '256',
    '--warmups', '1',
    '--repetitions', '2',
    '--repeat-checks', '1',
    '--output-dir', str(QUICK_RESULTS),
]
subprocess.run(quick_command, check=True, cwd=PROJECT)

In [ ]:
import pandas as pd
from IPython.display import Image, display

comparison = pd.read_csv(QUICK_RESULTS / 'comparison.csv')
display(comparison[[
    'workload', 'model_types', 'native_backend', 'framework_device',
    'threads', 'block_size', 'verification', 'max_abs_error',
    'native_mean_ms', 'framework_mean_ms', 'native_speedup',
]])
for plot in sorted((QUICK_RESULTS / 'plots').glob('*.png')):
    display(Image(filename=str(plot)))

## Optional long experiments

Set either flag to `True` and rerun the next cell. The million-node case can consume several gigabytes of host/GPU memory and substantial Colab time. It intentionally omits the sequential benchmark; validation falls back to the first requested native backend and still compares against PyTorch Geometric.

In [ ]:
RUN_CORA = False
RUN_MILLION_NODES = False

def run_optional(name, arguments):
    output = RESULTS / name
    command = [
        sys.executable, str(PROJECT / 'scripts/run_experiments.py'),
        '--native', str(EXECUTABLE),
        *arguments,
        '--output-dir', str(output),
    ]
    subprocess.run(command, check=True, cwd=PROJECT)
    return output

if RUN_CORA:
    run_optional('cora', [
        '--dataset', 'Cora', '--nodes', '1000', '--widths', '32',
        '--depths', '2', '--skews', '0',
        '--backend', 'sequential', 'parallel', 'cuda',
        '--threads', '1', '2', '--block-size', '128', '256',
        '--warmups', '2', '--repetitions', '10',
    ])

if RUN_MILLION_NODES:
    run_optional('million-nodes', [
        '--dataset', 'none', '--nodes', '1000000', '--widths', '32',
        '--depths', '2', '--skews', '0',
        '--backend', 'parallel', 'cuda', '--threads', '2',
        '--block-size', '128', '256', '--warmups', '1',
        '--repetitions', '5', '--repeat-checks', '0',
    ])

if not RUN_CORA and not RUN_MILLION_NODES:
    print('Optional experiments skipped. Set a flag above to True to run one.')